## Importación de las librerías

In [ ]:
# imports necesarios
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.mplot3d import Axes3D

from matplotlib.animation import FuncAnimation

In [ ]:
os.makedirs('images', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('./images/t-sne_thumbnails', exist_ok=True)
os.makedirs('gifs', exist_ok=True)

## Emplear la GPU

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## Descarga del dataset Food101

In [ ]:
# ----------------------------
# Data
# ----------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

trainset = torchvision.datasets.Food101(root="./data", split='train', download=True, transform=transform)
testset  = torchvision.datasets.Food101(root="./data", split='test', download=True, transform=transform)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

# Optional: keep only 4 classes (0,1,2,3)
keep_classes = [0, 1, 2, 3]
def filter_food101(dataset, keep):
    indices = [i for i, label in enumerate(dataset._labels) if label in keep]
    return Subset(dataset, indices)

trainset = filter_food101(trainset, keep_classes)
testset  = filter_food101(testset, keep_classes)

batch_size = 128
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
testloader  = DataLoader(testset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

## Modelos

### Encoder

In [ ]:
class FoodEncoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),  # 16x16
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(32, 64, 4, 2, 1), # 8x8
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(64, 128, 4, 2, 1), # 4x4
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, True),
        )
        self.fc_mu = nn.Linear(100352, latent_dim)
        self.fc_logvar = nn.Linear(100352, latent_dim)

    def forward(self, x):
        h = self.net(x).view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

### Decoder

In [ ]:
class FoodDecoder(nn.Module):
    def __init__(self, latent_dim=64):
        super(FoodDecoder, self).__init__()

        self.fc = nn.Linear(latent_dim, 100352) # 128 * 28 * 28
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        h = self.fc(z)
        h = h.view(h.size(0), 128, 28, 28)
        return self.net(h)


### VAE

In [ ]:
class FoodVAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.enc = FoodEncoder(latent_dim)
        self.dec = FoodDecoder(latent_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.enc(x)
        z = self.reparameterize(mu, logvar)
        recon = self.dec(z)
        return recon, mu, logvar


### Pérdida para el VAE

In [ ]:
# ----------------------------
# Loss (ELBO)
# ----------------------------
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    '''
    recon = F.mse_loss(recon_x, x, reduction="sum")
    kl = torch.mean(-0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp(), dim = 1), dim = 0)
    return recon + beta * kl, recon, kl
    '''
    # Debido a que durante el entrenamiento los valores de KL y de Rec se quedaron en Nan desde la época 5 hasta la 100 se han realizdo unos vcambios
    # respecto a la función de pérdida original
    batch_size = x.size(0)
    recon = F.mse_loss(recon_x, x, reduction="sum") / batch_size
    logvar = torch.clamp(logvar, min=-15.0, max=5.0)
    mu = torch.clamp(mu, min=-15.0, max=15.0)
    kl = torch.mean(-0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp(), dim=1))
    return recon + beta * kl, recon, kl

### Mostrar imágenes

In [ ]:
def _plot_training_results(history, title):
    def normalize(data):
        d = np.array(data)
        return (d - d.min()) / (d.max() - d.min() + 1e-8)

    ep_rng = range(1, len(history['loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # Subplot 1: Métricas Normalizadas
    axes[0].plot(ep_rng, normalize(history['rec']), label='Recon (Norm)', color='green')
    axes[0].plot(ep_rng, normalize(history['kl']), label='KL (Norm)', color='red')
    axes[0].plot(ep_rng, normalize(history['beta']), label='Beta (Evolución)', color='orange', linestyle='--', linewidth=3)
    axes[0].set_title("Dinámica Normalizada: Recon vs KL vs Beta")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Subplot 2: Reconstrucción Real
    axes[1].plot(ep_rng, history['rec'], label='MSE Real', color='green', linewidth=2)
    axes[1].set_title("Calidad de Reconstrucción (Sin normalizar)")
    axes[1].set_ylabel("Suma de Errores Cuadráticos")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(f'./images/{title}.png')
    plt.show()

### Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, path='checkpoint.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)

        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

### Training

In [ ]:
def train_beta_vae(model, optimizer, train_loader, val_loader,
                   epochs=100, cycles=4, max_beta=1.0,
                   patience=20, title="Food101_VAE"):

    history = {'loss': [], 'rec': [], 'kl': [], 'beta': []}
    early_stopping = EarlyStopping(patience=patience, path=f'./models/{title}_best.pth')

    device = next(model.parameters()).device

    # Parámetros para el ciclo
    period = epochs // cycles

    for ep in range(1, epochs + 1):
        # Cálculo de Beta Cíclico
        relative_pos = (ep - 1) % period / (period / 2)
        current_beta = max_beta * min(1.0, relative_pos)

        model.train()
        train_loss, train_rec, train_kl = 0.0, 0.0, 0.0

        for x, _ in train_loader:
            x = x.to(device)
            recon, mu, logvar = model(x)

            loss, rec, kl = vae_loss(recon, x, mu, logvar, beta=current_beta)

            optimizer.zero_grad()
            loss.backward()
            # Cortar el gradiente si supera un valor de 0.5
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

            train_loss += loss.item()
            train_rec += rec.item()
            train_kl += kl.item()

        # Métricas medias
        n = len(train_loader)
        history['loss'].append(train_loss / n)
        history['rec'].append(train_rec / n)
        history['kl'].append(train_kl / n)
        history['beta'].append(current_beta)

        print(f"Epoch {ep:02d}/{epochs} | Beta: {current_beta:.3f} | Rec: {history['rec'][-1]:.2f} | KL: {history['kl'][-1]:.2f}")


        model.eval()
        with torch.no_grad():
            x_test, _ = next(iter(val_loader))
            x_test = x_test[:16].to(device)
            recon_test, _, _ = model(x_test)

            def denorm(t): return (t * 0.5 + 0.5).clamp(0,1)

            x_img = denorm(x_test).cpu()
            recon_img = denorm(recon_test).cpu()

            fig, ax = plt.subplots(2, 16, figsize=(32, 4))
            for i in range(16):

                img_orig = x_img[i].permute(1, 2, 0).numpy()
                img_recon = recon_img[i].permute(1, 2, 0).numpy()


                ax[0, i].imshow(img_orig)
                ax[0, i].axis("off")

                ax[1, i].imshow(img_recon)
                ax[1, i].axis("off")

            fig.suptitle(f"Epoch {ep} - Reconstrucción paso a paso")
            plt.tight_layout()
            plt.show()

        val_loss_tot = 0
        val_rec_tot = 0

        model.eval()
        with torch.no_grad():

            for xv, _ in val_loader:
                xv = xv.to(device)
                rv, mv, lv = model(xv)

                v_loss, v_rec, v_kl = vae_loss(rv, xv, mv, lv, beta=current_beta)

                val_loss_tot += v_loss.item()
                val_rec_tot += v_rec.item()

        avg_val_loss = val_loss_tot / len(val_loader)
        avg_val_rec = val_rec_tot / len(val_loader)

        early_stopping(avg_val_rec, model)

        if early_stopping.early_stop:
            print("Early stopping activado. Recargando mejor modelo.")
            model.load_state_dict(torch.load(early_stopping.path))
            break

    _plot_training_results(history, title)

    return history

###  Pruebas Food 101 con diferentes espacios latentes

- Con 64

In [ ]:
latent_dim = 64
model_latent_dim_64 = FoodVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_64.parameters(), lr=1e-4)
epochs = 100
cycles = 4
patience = 20
max_beta = 1.0

results = train_beta_vae(
    model_latent_dim_64,
    opt,
    trainloader,
    testloader,
    epochs=epochs,
    cycles=cycles,
    max_beta=max_beta,
    patience=patience,
    title="Beta_VAE_ld_64"
)

- Con 48

In [ ]:
latent_dim = 48
model_latent_dim_48 = FoodVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_48.parameters(), lr=1e-4)
epochs = 100
cycles = 4
patience = 20
max_beta = 1.0

results = train_beta_vae(
    model_latent_dim_48,
    opt,
    trainloader,
    testloader,
    epochs=epochs,
    cycles=cycles,
    max_beta=max_beta,
    patience=patience,
    title="Beta_VAE_ld_48"
)

- Con 32

In [ ]:
latent_dim = 32
model_latent_dim_32 = FoodVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_32.parameters(), lr=1e-3)
epochs = 100
cycles = 4
patience = 20
max_beta = 1.0

results = train_beta_vae(
    model_latent_dim_32,
    opt,
    trainloader,
    testloader,
    epochs=epochs,
    cycles=cycles,
    max_beta=max_beta,
    patience=patience,
    title="Beta_VAE_ld_32"
)

- Con 16

In [ ]:
latent_dim = 16
model_latent_dim_16 = FoodVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_16.parameters(), lr=1e-3)
epochs = 100
cycles = 4
patience = 20
max_beta = 1.0

results = train_beta_vae(
    model_latent_dim_16,
    opt,
    trainloader,
    testloader,
    epochs=epochs,
    cycles=cycles,
    max_beta=max_beta,
    patience=patience,
    title="Beta_VAE_ld_16"
)

In [ ]:
- Con 8

In [ ]:
latent_dim = 8
model_latent_dim_8 = FoodVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_8.parameters(), lr=1e-3)
epochs = 100
cycles = 4
patience = 20
max_beta = 1.0

results = train_beta_vae(
    model_latent_dim_8,
    opt,
    trainloader,
    testloader,
    epochs=epochs,
    cycles=cycles,
    max_beta=max_beta,
    patience=patience,
    title="Beta_VAE_ld_8"
)

- Con 4

In [ ]:
latent_dim = 4
model_latent_dim_4 = FoodVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_4.parameters(), lr=1e-3)
epochs = 100
cycles = 4
patience = 20
max_beta = 1.0

results = train_beta_vae(
    model_latent_dim_4,
    opt,
    trainloader,
    testloader,
    epochs=epochs,
    cycles=cycles,
    max_beta=max_beta,
    patience=patience,
    title="Beta_VAE_ld_4"
)

### Cargar los pesos de los modelos

- Con 64

In [ ]:
latent_dim = 64
modelo_ld_64 = FoodVAE(latent_dim).to(device)

modelo_ld_64.load_state_dict(torch.load("./models/Beta_VAE_ld_64_best.pth", map_location=device))

modelo_ld_64.eval()

- Con 48

In [ ]:
latent_dim = 48
modelo_ld_48 = FoodVAE(latent_dim).to(device)

modelo_ld_48.load_state_dict(torch.load("./models/Beta_VAE_ld_48_best.pth", map_location=device))

modelo_ld_48.eval()

- Con 32

In [ ]:
latent_dim = 32
modelo_ld_32 = FoodVAE(latent_dim).to(device)

modelo_ld_32.load_state_dict(torch.load("./models/Beta_VAE_ld_32_best.pth", map_location=device))

modelo_ld_32.eval()

- Con 8

In [ ]:
latent_dim = 8
modelo_ld_8 = FoodVAE(latent_dim).to(device)

modelo_ld_8.load_state_dict(torch.load("./models/Beta_VAE_ld_8_best.pth", map_location=device))

modelo_ld_8.eval()

- Con 4

In [ ]:
latent_dim = 4
modelo_ld_4 = FoodVAE(latent_dim).to(device)

modelo_ld_4.load_state_dict(torch.load("./models/Beta_VAE_ld_4_best.pth", map_location=device))

modelo_ld_4.eval()

## Interpolación

In [ ]:
# Interpolation
# Para ello tomamos 2 imágenes del test set, las codificamos, y luego interpolamos linealmente en el espacio latente entre sus representaciones.
# Finalmente decodificamos cada punto de la interpolación para ver la transición entre ambas imágenes.
import torch
import matplotlib.pyplot as plt

def interpolation(model):

    with torch.no_grad():
        model.eval()
        x, _ = next(iter(DataLoader(testset, batch_size=2)))
        x = x[:2].to(device)
        mu, logvar = model.enc(x)
        z = model.reparameterize(mu, logvar)
        z1, z2 = z[0], z[1]
        n_interp = 15
        interp_z = torch.stack([z1 * (1 - alpha) + z2 * alpha for alpha in torch.linspace(0, 1, n_interp)], dim=0)
        interp_x = model.dec(interp_z)

        def denorm(t):
            return (t * 0.5 + 0.5).clamp(0, 1)

        interp_x = denorm(interp_x).cpu()
        fig, ax = plt.subplots(1, n_interp, figsize=(2 * n_interp, 3))
        for i in range(n_interp):
            img = interp_x[i].permute(1, 2, 0).numpy()
            ax[i].imshow(img)
            ax[i].axis("off")

        fig.suptitle("Interpolación en el Espacio Latente (Food101)", fontsize=16)
        plt.tight_layout()
        plt.show()


- Con 64

In [ ]:
interpolation(model=modelo_ld_64)

- Con 48

In [ ]:
interpolation(model=modelo_ld_48)

- Con 32

In [ ]:
interpolation(model=modelo_ld_32)

- Con 16

In [ ]:
interpolation(model=modelo_ld_16)

- Con 8

In [ ]:
interpolation(model=modelo_ld_8)

- Con 4

In [ ]:
interpolation(model=modelo_ld_4)

## Espacio Latente 2D con t-SNE

In [ ]:
def denorm(t): return (t * 0.5 + 0.5).clamp(0,1)

def latent_space(model, title_p1="Latent_Test", title_p2="Latent_Prior"):
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        latents = []
        labels = []
        for x, y in DataLoader(testset, batch_size=128):
            x = x.to(device)
            _, mu, _ = model(x)
            latents.append(mu.cpu())
            labels.append(y)
        latents = torch.cat(latents, dim=0).numpy()
        labels = torch.cat(labels, dim=0).numpy()

    tsne = TSNE(n_components=2, random_state=42)
    latents_2d = tsne.fit_transform(latents)

    plt.figure(figsize=(8, 8))
    unique_classes = np.unique(labels)
    for digit in unique_classes:
        idx = labels == digit
        plt.scatter(latents_2d[idx, 0], latents_2d[idx, 1], label=str(digit), alpha=0.5)
    plt.legend()
    plt.title(title_p1)
    plt.savefig(f'./images/{title_p1}.png')
    plt.show()

    with torch.no_grad():
        model.eval()
        n_samples = 1000
        latent_dim = model.enc.fc_mu.out_features
        z = torch.randn(n_samples, latent_dim, device=device)
        recon = model.dec(z)
        recon = denorm(recon).cpu()
        recon = recon.view(n_samples, 3, 224, 224)
        recon = recon.numpy()
        recon_tensor = torch.from_numpy(recon).float().to(device)
        latents = model.enc.net(recon_tensor).view(n_samples, -1).detach().cpu().numpy()
        tsne = TSNE(n_components=2, random_state=42)
        latents_2d = tsne.fit_transform(latents)
        plt.figure(figsize=(8, 8))
        plt.scatter(latents_2d[:, 0], latents_2d[:, 1], alpha=0.5)
        plt.title(title_p2)
        plt.savefig(f'./images/{title_p2}.png')
        plt.show()

- Con 64

In [ ]:
latent_space(model=modelo_ld_64, title_p1='t-SNE of beta VAE Food101 Latent Space 64', title_p2='t-SNE of Prior Samples in Latent Space 64 beta VAE Food101')

- Con 48

In [ ]:
latent_space(model=modelo_ld_48, title_p1='t-SNE of beta VAE Food101 Latent Space 48', title_p2='t-SNE of Prior Samples in Latent Space 48 beta VAE Food101')

- Con 32

In [ ]:
latent_space(model=modelo_ld_32, title_p1='t-SNE of beta VAE Food101 Latent Space 32', title_p2='t-SNE of Prior Samples in Latent Space 32 beta VAE Food101')

- Con 16

In [ ]:
latent_space(model=modelo_ld_16, title_p1='t-SNE of beta VAE Food101 Latent Space 16', title_p2='t-SNE of Prior Samples in Latent Space 16 beta VAE Food101')

- Con 8

In [ ]:
latent_space(model=modelo_ld_8, title_p1='t-SNE of beta VAE Food101 Latent Space 8', title_p2='t-SNE of Prior Samples in Latent Space 8 beta VAE Food101')

- Con 4

In [ ]:
latent_space(model=modelo_ld_4, title_p1='t-SNE of beta VAE Food101 Latent Space 4', title_p2='t-SNE of Prior Samples in Latent Space 4 beta VAE Food101')

## t-SNE 3D

In [ ]:
def generar_y_descargar_gif(model, device, testset, titulo):
    model.eval()
    latents = []
    labels = []

    with torch.no_grad():
        for x, y in DataLoader(testset, batch_size=128):
            x = x.to(device)
            _, mu, _ = model(x)
            latents.append(mu.cpu())
            labels.append(y)
        latents = torch.cat(latents, dim=0).numpy()
        labels = torch.cat(labels, dim=0).numpy()

    n_samples = min(len(latents), 2000)
    indices = np.random.choice(len(latents), n_samples, replace=False)
    latents_sample = latents[indices]
    labels_sample = labels[indices]

    tsne = TSNE(n_components=3, random_state=42)
    latents_3d = tsne.fit_transform(latents_sample)

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    unique_classes = np.unique(labels_sample)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_classes)))

    def update(frame):
        ax.clear()
        ax.set_box_aspect([1,1,1])
        ax.set_axis_off()

        for idx_color, clase in enumerate(unique_classes):
            idx = labels_sample == clase
            ax.scatter(latents_3d[idx, 0],
                       latents_3d[idx, 1],
                       latents_3d[idx, 2],
                       c=[colors[idx_color]],
                       label=f'Clase {clase}',
                       s=15,
                       alpha=0.6)

        ax.set_title(titulo)
        ax.legend(loc='upper left', bbox_to_anchor=(1, 0.9), title="Comida")
        ax.view_init(elev=20, azim=frame)
        return fig

    frames_lentos = np.arange(0, 360, 2)
    ani = FuncAnimation(fig, update, frames=frames_lentos, interval=50)

    # Guardado
    ani.save(f'./gifs/{titulo}.gif', writer='pillow', fps=20)
    plt.close()

- Con 64

In [ ]:
generar_y_descargar_gif(modelo_ld_64, device, testset, titulo = 't_SNE_3D_ld_64')

- Con 48

In [ ]:
generar_y_descargar_gif(modelo_ld_48, device, testset, titulo = 't_SNE_3D_ld_48')

- Con 32

In [ ]:
generar_y_descargar_gif(modelo_ld_32, device, testset, titulo = 't_SNE_3D_ld_32')

- Con 16

In [ ]:
generar_y_descargar_gif(modelo_ld_16, device, testset, titulo = 't_SNE_3D_ld_16')

- Con 8

In [ ]:
generar_y_descargar_gif(modelo_ld_8, device, testset, titulo = 't_SNE_3D_ld_8')

- Con 4

In [ ]:
generar_y_descargar_gif(modelo_ld_4, device, testset, titulo = 't_SNE_3D_ld_4')

## t-SNE con Reconstrucciones

In [ ]:
@torch.no_grad()
def denorm_food101(t):  # [-1,1] -> [0,1]
    return (t * 0.5 + 0.5).clamp(0, 1)

def pick_spread_points(emb, n_show=150, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(emb.shape[0])
    rng.shuffle(idx)

    chosen = []
    thr = 0.02 * np.var(emb, axis=0).sum()
    for i in idx:
        if not chosen:
            chosen.append(i); continue
        d2 = np.min(np.sum((emb[chosen] - emb[i])**2, axis=1))
        if d2 > thr:
            chosen.append(i)
        if len(chosen) >= n_show:
            break
    return np.array(chosen)

@torch.no_grad()
def tsne_with_recon_thumbnails(model, loader, device, n_total=3000, n_show=150, zoom=0.08, seed=42, title=None):
    model.eval()

    X = []
    MU = []
    RECON = []

    for x, _ in loader:
        x = x.to(device)
        recon, mu, logvar = model(x)
        X.append(x.detach().cpu())
        MU.append(mu.detach().cpu())
        RECON.append(recon.detach().cpu())
        if sum(t.size(0) for t in MU) >= n_total:
            break

    X = torch.cat(X, dim=0)[:n_total]
    MU = torch.cat(MU, dim=0)[:n_total].numpy()
    RECON = torch.cat(RECON, dim=0)[:n_total]

    emb = TSNE(n_components=2, random_state=seed, init="pca", learning_rate="auto").fit_transform(MU)
    chosen = pick_spread_points(emb, n_show=n_show, seed=seed)

    thumbs = denorm_food101(RECON[chosen]).permute(0, 2, 3, 1).numpy()

    fig, ax = plt.subplots(figsize=(10, 10))
    if title:
        ax.set_title(f"{title}")
    ax.set_xticks([]); ax.set_yticks([])

    for k, i in enumerate(chosen):
        ab = AnnotationBbox(OffsetImage(thumbs[k], zoom=zoom),
                            (emb[i,0], emb[i,1]), frameon=False)
        ax.add_artist(ab)

    ax.set_xlim(emb[chosen,0].min()-5, emb[chosen,0].max()+5)
    ax.set_ylim(emb[chosen,1].min()-5, emb[chosen,1].max()+5)
    plt.tight_layout()
    plt.savefig(f'./images/t-sne_thumbnails/{title}.png')
    plt.show()
testloader_tsne = DataLoader(testset, batch_size=128, shuffle=False)


In [ ]:
tsne_with_recon_thumbnails(modelo_ld_64, testloader_tsne, device, n_total=3000, n_show=150, zoom=0.08, title='t-SNE of beta VAE Food LS 64 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_48, testloader_tsne, device, n_total=3000, n_show=150, zoom=0.08, title='t-SNE of beta VAE Food LS 48 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_32, testloader_tsne, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of beta VAE Food LS 32 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_16, testloader_tsne, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of beta VAE Food LS 16 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_8, testloader_tsne, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of beta VAE Food LS 8 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_4, testloader_tsne, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of beta VAE Food LS 4 Img')